In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS weather_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_catalog.bronze")

In [0]:
import os
import uuid
import importlib
import sys
from src.ingestion.api_client import WeatherApiClient
from src.ingestion.bronze_writer import save_raw_json_to_bronze

API_KEY = "40182f92088b4a239ce152535261908"
client = WeatherApiClient(api_key=API_KEY)
batch_id = str(uuid.uuid4())
cities = ["London", "New York", "Tokyo", "Paris", "Sydney", "Kolkata"]

raw_observations = []
for city in cities:
    data = client.fetch_current(city)
    raw_observations.append(data)

save_raw_json_to_bronze(
    spark=spark,
    raw_payloads=raw_observations,
    table_name="weather_catalog.bronze.brz_weather_observations",
    batch_id=batch_id
)

In [0]:
df = spark.read.table("weather_catalog.bronze.brz_weather_observations")
display(df)
df.printSchema()